# Contextual Rejection Model

## An independent answer to “which track and artist do I reject most?”

This notebook does **not** reuse the prior skip-rate or completion-ratio rankings. It works directly from every `Streaming_History_*.json` export and uses Rewind’s enriched `track_features` metadata, treating “hate” as a behavioral inference rather than a fact Spotify records.

### The idea

A strong negative signal is an **explicit forward skip before meaningful completion**. Metadata makes “meaningful” relative to each track’s real catalog duration rather than a fixed number of seconds. The signal is only meaningful in context:

- A track started with `clickrow` or `playbtn` was deliberately selected, so it is not used as a forced-exposure trial.
- Passive trials are limited to known continuation, skip-through, and app-resume starts.
- The model compares tracks only against plays with the same start mechanism, shuffle state, and duration band.
- It collapses repeated events within a listening session, so a skip spiral cannot become dozens of independent votes.
- Deliberately selecting an entity is counterevidence, not proof of dislike.
- A track needs statistical evidence and minimum support before it can win.
- An artist is reported only when the evidence spans multiple rejected tracks.

The result is an auditable *contextual rejection* ranking. It can say “the data are insufficient” rather than manufacture certainty.

In [1]:
# This notebook uses the enriched per-session metadata already created by Rewind.
# DuckDB is a project dependency in backend/.venv; select that kernel if this import fails.
from collections import Counter, defaultdict
from math import erfc, sqrt
from pathlib import Path
import json
import random
import statistics
import unicodedata

import duckdb


def find_project_root():
    """Find the directory that owns the raw Spotify exports."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project's data directory.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
SESSION_DB = DATA_DIR / "sessions" / "rewind.duckdb"
EXPORT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "contextual_rejection"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if not SESSION_DB.is_file():
    raise FileNotFoundError(
        f"Missing enriched session database: {SESSION_DB}. "
        "Upload the exports in Rewind once so track_features is created."
    )

CONFIG = {
    "rapid_completion": 0.30,
    "choice_starts": frozenset({"clickrow", "playbtn"}),
    "clear_ends": frozenset({"trackdone", "fwdbtn"}),
    "session_gap_minutes": 30,
    "prior_strength": 20.0,
    "track_min_exposures": 10,
    "track_min_rejections": 4,
    "artist_min_exposures": 30,
    "artist_min_rejections": 8,
    "artist_min_tracks": 2,
    "track_min_excess_rate": 0.10,
    "artist_min_excess_rate": 0.08,
    "max_false_discovery_rate": 0.05,
    "bootstrap_samples": 250,
    "random_seed": 20260911,
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Metadata source: {SESSION_DB}")
print(f"Exports will be written to: {EXPORT_DIR}")
print("Rapid rejection: forward skip before 30% of the catalog duration")

Project root: /home/kala/Documents/rewind
Metadata source: /home/kala/Documents/rewind/data/sessions/rewind.duckdb
Exports will be written to: /home/kala/Documents/rewind/notebooks/outputs/contextual_rejection
Rapid rejection: forward skip before 30% of the catalog duration


## 1. Load and Audit the Music Dataset

The notebook reads every `Streaming_History_*.json` file, including video history when it contains music tracks. It keeps only rows with a Spotify track URI, identifies exact duplicate export records, and prints a compact schema and missing-value audit.

## 2. Standardize Track and Artist Identifiers

Spotify track URIs are the primary track identifier. Text normalization exists only as a fallback and to merge harmless artist spelling differences such as accents, case, punctuation, and `feat.` suffixes. Original display text is always preserved.

## 3. Profile Available “Dislike” Signals

The export contains no explicit thumbs-down or rating field. The usable behavioral evidence is therefore:

| Field | Role in the model |
| --- | --- |
| `reason_end == "fwdbtn"` | Explicit forward-skip action |
| `ms_played` | Separates a rapid rejection from a late skip |
| `reason_start` | Distinguishes deliberate starts from autoplay / continuation contexts |
| `shuffle` | Keeps shuffled and unshuffled exposure patterns comparable |
| `skipped` | Audit-only cross-check; not used as the primary label |

A rapid forward skip means `fwdbtn` within the configurable window. A `backbtn` is not treated as dislike because it can mean replaying the preceding track.

In [2]:
from datetime import datetime


def track_id_from_uri(track_uri):
    if not track_uri:
        return None
    return track_uri.rsplit(":", 1)[-1]


source_files = sorted(DATA_DIR.glob("Streaming_History_*.json"))
if not source_files:
    raise FileNotFoundError(f"No Streaming_History_*.json files found in {DATA_DIR}")

raw_rows = []
for path in source_files:
    with path.open(encoding="utf-8") as handle:
        part = json.load(handle)
    raw_rows.extend(part)
    print(f"{path.name}: {len(part):,} rows")

music_rows = [row for row in raw_rows if row.get("spotify_track_uri")]
seen = set()
unique_music_rows = []
for row in music_rows:
    fingerprint = (
        row.get("ts"), row.get("spotify_track_uri"), row.get("ms_played"),
        row.get("reason_start"), row.get("reason_end"),
    )
    if fingerprint not in seen:
        seen.add(fingerprint)
        unique_music_rows.append(row)

duplicate_count = len(music_rows) - len(unique_music_rows)
all_columns = sorted({column for row in raw_rows for column in row})
missing = {
    column: sum(row.get(column) in (None, "") for row in music_rows)
    for column in all_columns
}

with duckdb.connect(str(SESSION_DB), read_only=True) as metadata_con:
    table_exists = metadata_con.execute(
        "SELECT COUNT(*) FROM information_schema.tables WHERE table_name = 'track_features'"
    ).fetchone()[0]
    if not table_exists:
        raise RuntimeError("The session database has no track_features table. Re-upload the exports first.")
    feature_rows = metadata_con.execute(
        """
        SELECT
            track_id, duration, artist_id, artist_name, artist_genres, popularity,
            release_year, danceability, energy, valence
        FROM track_features
        """
    ).fetchall()

metadata_by_id = {
    track_id: {
        "duration_s": duration,
        "catalog_artist_id": artist_id,
        "catalog_artist": catalog_artist,
        "genres": genres,
        "popularity": popularity,
        "release_year": release_year,
        "danceability": danceability,
        "energy": energy,
        "valence": valence,
    }
    for (
        track_id, duration, artist_id, catalog_artist, genres, popularity,
        release_year, danceability, energy, valence,
    ) in feature_rows
}

matched_events = sum(
    track_id_from_uri(row.get("spotify_track_uri")) in metadata_by_id
    for row in unique_music_rows
)
usable_duration_events = sum(
    bool(metadata_by_id.get(track_id_from_uri(row.get("spotify_track_uri")), {}).get("duration_s"))
    for row in unique_music_rows
)
unique_track_ids = {track_id_from_uri(row.get("spotify_track_uri")) for row in unique_music_rows}
matched_tracks = sum(track_id in metadata_by_id for track_id in unique_track_ids)

print(f"\nAll export rows: {len(raw_rows):,}")
print(f"Music rows: {len(music_rows):,}")
print(f"Exact duplicate music rows removed: {duplicate_count:,}")
print(f"Available columns ({len(all_columns)}): {', '.join(all_columns)}")
print("\nMetadata join audit:")
print(f"  Event coverage: {matched_events:,} / {len(unique_music_rows):,} ({matched_events / len(unique_music_rows):.1%})")
print(f"  Usable duration: {usable_duration_events:,} / {len(unique_music_rows):,} ({usable_duration_events / len(unique_music_rows):.1%})")
print(f"  Track coverage: {matched_tracks:,} / {len(unique_track_ids):,} ({matched_tracks / len(unique_track_ids):.1%})")
print("  Unmatched events are excluded from duration-normalized scoring, not imputed.")

print("\nMost-missing fields among music rows:")
for column, count in sorted(missing.items(), key=lambda item: item[1], reverse=True)[:12]:
    print(f"  {column}: {count:,} / {len(music_rows):,} missing")

print("\nRepresentative enriched music event:")
example = unique_music_rows[0]
example_metadata = metadata_by_id.get(track_id_from_uri(example.get("spotify_track_uri")), {})
for key in (
    "ts", "spotify_track_uri", "master_metadata_track_name",
    "master_metadata_album_artist_name", "ms_played", "reason_start",
    "reason_end", "shuffle", "skipped",
):
    print(f"  {key}: {example.get(key)!r}")
print(f"  catalog duration (seconds): {example_metadata.get('duration_s')!r}")
print(f"  catalog genres: {example_metadata.get('genres')!r}")

Streaming_History_Audio_2022-2025_0.json: 16,168 rows
Streaming_History_Audio_2025_1.json: 1,480 rows
Streaming_History_Video_2024.json: 155 rows

All export rows: 17,803
Music rows: 17,801
Exact duplicate music rows removed: 61
Available columns (23): audiobook_chapter_title, audiobook_chapter_uri, audiobook_title, audiobook_uri, conn_country, episode_name, episode_show_name, incognito_mode, ip_addr, master_metadata_album_album_name, master_metadata_album_artist_name, master_metadata_track_name, ms_played, offline, offline_timestamp, platform, reason_end, reason_start, shuffle, skipped, spotify_episode_uri, spotify_track_uri, ts

Metadata join audit:
  Event coverage: 16,855 / 17,740 (95.0%)
  Usable duration: 16,855 / 17,740 (95.0%)
  Track coverage: 2,730 / 2,972 (91.9%)
  Unmatched events are excluded from duration-normalized scoring, not imputed.

Most-missing fields among music rows:
  audiobook_chapter_title: 17,801 / 17,801 missing
  audiobook_chapter_uri: 17,801 / 17,801 missi

In [8]:
def normalize_text(value):
    """Conservative fallback key for display-name variants."""
    text = (value or "").strip()
    text = text.split(" (feat.", 1)[0].split(" (with ", 1)[0]
    text = unicodedata.normalize("NFKD", text).lower()
    text = "".join(char for char in text if not unicodedata.combining(char))
    return "".join(char for char in text if char.isalnum())


def most_common_label(counter, fallback):
    if not counter:
        return fallback
    return sorted(counter.items(), key=lambda item: (-item[1], item[0]))[0][0]


def parse_export_timestamp(value):
    return datetime.fromisoformat(str(value).replace("Z", "+00:00"))


def duration_band(duration_s):
    if duration_s < 120:
        return "under-2m"
    if duration_s < 210:
        return "2m-to-3m30"
    if duration_s < 300:
        return "3m30-to-5m"
    return "5m-plus"


def start_bucket(reason_start):
    if reason_start == "trackdone":
        return "continuation"
    if reason_start == "fwdbtn":
        return "skip-through"
    if reason_start == "appload":
        return "resume"
    return reason_start


ordered_rows = sorted(unique_music_rows, key=lambda row: parse_export_timestamp(row["ts"]))
plays = []
previous_timestamp = None
session_number = 0

for row in ordered_rows:
    timestamp = parse_export_timestamp(row["ts"])
    if (
        previous_timestamp is not None
        and (timestamp - previous_timestamp).total_seconds() > CONFIG["session_gap_minutes"] * 60
    ):
        session_number += 1
    previous_timestamp = timestamp

    track_id = track_id_from_uri(row.get("spotify_track_uri"))
    metadata = metadata_by_id.get(track_id)
    duration_s = metadata.get("duration_s") if metadata else None
    duration_ms = int(duration_s * 1000) if duration_s else None
    ms_played = int(row.get("ms_played") or 0)
    completion = min(ms_played / duration_ms, 1.0) if duration_ms else None
    reason_start = row.get("reason_start") or "unknown"
    reason_end = row.get("reason_end") or "unknown"
    export_artist = row.get("master_metadata_album_artist_name") or "Unknown artist"
    catalog_artist_id = metadata.get("catalog_artist_id") if metadata else None

    plays.append({
        "session": session_number,
        "timestamp": timestamp,
        "track_id": track_id,
        "track_key": track_id or f"text:{normalize_text(row.get('master_metadata_track_name'))}::{normalize_text(export_artist)}",
        "artist_key": f"id:{catalog_artist_id}" if catalog_artist_id else f"text:{normalize_text(export_artist)}",
        "track": row.get("master_metadata_track_name") or "Unknown track",
        "artist": export_artist,
        "catalog_artist": metadata.get("catalog_artist") if metadata else None,
        "duration_s": duration_s,
        "completion": completion,
        "ms_played": ms_played,
        "reason_start": reason_start,
        "reason_end": reason_end,
        "shuffle": bool(row.get("shuffle")),
        "chosen": reason_start in CONFIG["choice_starts"],
        "passively_served": reason_start in {"trackdone", "fwdbtn", "appload"},
        "genres": metadata.get("genres") if metadata else None,
        "popularity": metadata.get("popularity") if metadata else None,
        "release_year": metadata.get("release_year") if metadata else None,
        "energy": metadata.get("energy") if metadata else None,
        "valence": metadata.get("valence") if metadata else None,
    })

for play in plays:
    if play["duration_s"]:
        play["duration_band"] = duration_band(play["duration_s"])
        play["context"] = (
            f"{start_bucket(play['reason_start'])}"
            f"|shuffle={int(play['shuffle'])}"
            f"|duration={play['duration_band']}"
        )
        play["rapid_rejection"] = (
            play["passively_served"]
            and play["reason_end"] == "fwdbtn"
            and play["completion"] <= CONFIG["rapid_completion"]
        )
        # A near-complete listen overrides a same-session skip loop as positive evidence.
        play["durable_acceptance"] = play["completion"] >= 0.80
    else:
        play["duration_band"] = None
        play["context"] = None
        play["rapid_rejection"] = False
        play["durable_acceptance"] = False

metadata_plays = [play for play in plays if play["duration_s"]]
completion_values = sorted(play["completion"] for play in metadata_plays)

def percentile(values, fraction):
    if not values:
        return None
    index = round((len(values) - 1) * fraction)
    return values[index]

print(f"Analysis events after deduplication: {len(plays):,}")
print(f"Listening sessions (>{CONFIG['session_gap_minutes']} min gap): {session_number + 1:,}")
print(f"Events with catalog duration: {len(metadata_plays):,} / {len(plays):,} ({len(metadata_plays) / len(plays):.1%})")
print(
    "Completion fractions (duration-normalized): "
    + ", ".join(
        f"p{int(p * 100)}={percentile(completion_values, p):.0%}"
        for p in (0.25, 0.50, 0.75)
    )
)
print("\nStart reasons:")
for value, count in Counter(play["reason_start"] for play in plays).most_common():
    print(f"  {value:>12}: {count:>6,}")
print("\nEnd reasons:")
for value, count in Counter(play["reason_end"] for play in plays).most_common():
    print(f"  {value:>12}: {count:>6,}")
print(
    "\nDirectly chosen events (clickrow/playbtn): "
    f"{sum(play['chosen'] for play in metadata_plays):,} / {len(metadata_plays):,}"
)
print(
    "Passively served rapid rejections before session collapsing: "
    f"{sum(play['rapid_rejection'] for play in metadata_plays):,}"
)
print(
    "Why duration matters: "
    f"{sum(play['ms_played'] <= 30_000 and play['completion'] > CONFIG['rapid_completion'] for play in metadata_plays):,} "
    "events are under 30 seconds but exceed the relative rejection threshold."
)

Analysis events after deduplication: 17,740
Listening sessions (>30 min gap): 1,095
Events with catalog duration: 16,855 / 17,740 (95.0%)
Completion fractions (duration-normalized): p25=16%, p50=100%, p75=100%

Start reasons:
     trackdone:  8,762
      clickrow:  3,913
        fwdbtn:  3,315
       backbtn:    803
       playbtn:    485
       appload:    254
       unknown:    157
    trackerror:     33
        remote:     18

End reasons:
     trackdone:  8,849
       endplay:  3,737
        fwdbtn:  3,293
       backbtn:    794
        logout:    573
  unexpected-exit-while-paused:    374
  unexpected-exit:     95
        remote:     17
    trackerror:      5
       unknown:      3

Directly chosen events (clickrow/playbtn): 4,148 / 16,855
Passively served rapid rejections before session collapsing: 1,854
Why duration matters: 4 events are under 30 seconds but exceed the relative rejection threshold.


## 4. Convert Events into Independent Exposure Trials

A Spotify export logs events, not independent opinions. One accidental autoplay loop can therefore create dozens of forward-skip events in a single sitting.

For the primary model, one **track-session** is one trial:

- The track must have catalog duration metadata.
- Any directly chosen start (`clickrow` or `playbtn`) removes that track-session from passive-exposure scoring.
- A trial is a **rejection** only if a passive play ends in an explicit forward skip before 30% completion, with no later durable listen in that same session.
- A trial is an **acceptance** if the listener reaches at least 80% completion in that session.
- All other partial or interrupted events are deliberately left unlabeled.

This is conservative on purpose: it tries to identify recurring, unprompted rejection rather than ordinary listening noise.

In [9]:
def make_session_trials(grouped_events):
    """Collapse all events for one entity in one listening session into one outcome."""
    trials = []
    seen_sessions = Counter()
    chosen_sessions = Counter()

    for (session, entity_key), events in grouped_events.items():
        seen_sessions[entity_key] += 1
        if any(event["chosen"] for event in events):
            # An intentional start anywhere in this entity-session is counterevidence.
            chosen_sessions[entity_key] += 1
            continue

        passive_events = [event for event in events if event["passively_served"]]
        durable_events = [event for event in passive_events if event["durable_acceptance"]]
        rejection_events = [event for event in passive_events if event["rapid_rejection"]]

        if durable_events:
            label = 0
            anchor = max(durable_events, key=lambda event: (event["completion"], event["timestamp"]))
        elif rejection_events:
            label = 1
            anchor = min(rejection_events, key=lambda event: event["timestamp"])
        else:
            continue

        trial = dict(anchor)
        trial["entity_key"] = entity_key
        trial["label"] = label
        trial["event_count"] = len(events)
        trials.append(trial)

    return trials, seen_sessions, chosen_sessions


track_session_events = defaultdict(list)
artist_session_events = defaultdict(list)
for play in metadata_plays:
    track_session_events[(play["session"], play["track_key"])].append(play)
    artist_session_events[(play["session"], play["artist_key"])].append(play)

track_trials, track_seen_sessions, track_chosen_sessions = make_session_trials(track_session_events)
artist_trials, artist_seen_sessions, artist_chosen_sessions = make_session_trials(artist_session_events)

assert len({(trial["session"], trial["entity_key"]) for trial in track_trials}) == len(track_trials)
assert len({(trial["session"], trial["entity_key"]) for trial in artist_trials}) == len(artist_trials)
assert {trial["label"] for trial in track_trials}.issubset({0, 1})
assert {trial["label"] for trial in artist_trials}.issubset({0, 1})

rejected_track_keys_by_artist = defaultdict(set)
for trial in track_trials:
    if trial["label"]:
        rejected_track_keys_by_artist[trial["artist_key"]].add(trial["track_key"])

print("Track-level independent trials:")
print(f"  Track-session groups: {len(track_session_events):,}")
print(f"  Groups excluded by a direct start: {sum(track_chosen_sessions.values()):,}")
print(f"  Labeled trials: {len(track_trials):,}")
print(f"  Rejections: {sum(trial['label'] for trial in track_trials):,}")
print(f"  Acceptances: {sum(not trial['label'] for trial in track_trials):,}")
print("\nArtist-level independent trials:")
print(f"  Artist-session groups: {len(artist_session_events):,}")
print(f"  Groups excluded by a direct start: {sum(artist_chosen_sessions.values()):,}")
print(f"  Labeled trials: {len(artist_trials):,}")
print(f"  Rejections: {sum(trial['label'] for trial in artist_trials):,}")
print(f"  Acceptances: {sum(not trial['label'] for trial in artist_trials):,}")
print(
    "\nEvent-to-session reduction: "
    f"{sum(play['rapid_rejection'] for play in metadata_plays):,} rapid events -> "
    f"{sum(trial['label'] for trial in track_trials):,} independent track-session rejections"
)

Track-level independent trials:
  Track-session groups: 14,004
  Groups excluded by a direct start: 3,695
  Labeled trials: 7,950
  Rejections: 1,648
  Acceptances: 6,302

Artist-level independent trials:
  Artist-session groups: 9,778
  Groups excluded by a direct start: 2,980
  Labeled trials: 5,372
  Rejections: 1,096
  Acceptances: 4,276

Event-to-session reduction: 1,854 rapid events -> 1,648 independent track-session rejections


## 5. Primary Model: Contextual Excess Rejection

For every independent trial, the model estimates how often a comparable passive play is rejected. Comparability is defined by:

- how the track started: continuation, skip-through, or app resume;
- whether shuffle was on; and
- a catalog-duration band, so a 90-second track is not judged against an 8-minute track.

For an entity $e$ in context $c$, the baseline is a leave-one-entity-out smoothed rejection rate:

$$
q_{e,c} = \frac{r_{-e,c} + s\,\bar r}{n_{-e,c} + s}
$$

where $r_{-e,c}$ and $n_{-e,c}$ exclude the entity being evaluated, $\bar r$ is the overall trial rejection rate, and $s=20$ provides light stabilization for sparse contexts.

The central signal is its excess rejection rate:

$$
\operatorname{excess}(e) = \frac{\sum_i(y_i-q_{e,c_i})}{n_e}
$$

A one-sided normal-tail screen and Benjamini-Hochberg false-discovery adjustment flag signals unlikely to be ordinary context noise. The final ranking multiplies positive contextual excess by $1-\text{direct-choice rate}$: a song or artist repeatedly selected on purpose is counterevidence against a blanket “hated” interpretation.

Metadata is used directly for duration normalization and to describe/audit finalists. Genre, popularity, and audio descriptors are **not** treated as causes of dislike.

In [10]:
def normal_survival(z_value):
    """One-sided standard-normal tail probability without extra dependencies."""
    return 0.5 * erfc(z_value / sqrt(2))


def benjamini_hochberg(rows):
    """Attach monotone Benjamini-Hochberg q-values to score rows in place."""
    ordered = sorted(enumerate(rows), key=lambda item: item[1]["p_value"])
    total = len(ordered)
    smallest_future_q = 1.0
    for rank, (index, row) in reversed(list(enumerate(ordered, start=1))):
        adjusted = min(smallest_future_q, row["p_value"] * total / rank, 1.0)
        rows[index]["q_value"] = adjusted
        smallest_future_q = adjusted


def score_contextual_rejection(trials, seen_sessions, chosen_sessions):
    """Score entities against their own leave-one-entity-out context baseline."""
    global_rejection_rate = sum(trial["label"] for trial in trials) / len(trials)
    context_n = Counter(trial["context"] for trial in trials)
    context_r = Counter(
        trial["context"] for trial in trials if trial["label"]
    )
    entity_context_n = defaultdict(Counter)
    entity_context_r = defaultdict(Counter)
    entity_n = Counter()
    entity_r = Counter()
    display_names = defaultdict(Counter)
    display_artists = defaultdict(Counter)

    for trial in trials:
        key = trial["entity_key"]
        context = trial["context"]
        entity_context_n[key][context] += 1
        entity_context_r[key][context] += trial["label"]
        entity_n[key] += 1
        entity_r[key] += trial["label"]
        display_names[key][trial["track"] if trial["track_key"] == key else (trial["catalog_artist"] or trial["artist"])] += 1
        display_artists[key][trial["artist"]] += 1

    scored = []
    for key, n_trials in entity_n.items():
        expected_rejections = 0.0
        null_variance = 0.0
        for context, own_n in entity_context_n[key].items():
            other_n = context_n[context] - own_n
            other_r = context_r[context] - entity_context_r[key][context]
            baseline = (
                other_r + CONFIG["prior_strength"] * global_rejection_rate
            ) / (other_n + CONFIG["prior_strength"])
            expected_rejections += own_n * baseline
            null_variance += own_n * baseline * (1.0 - baseline)

        rejections = entity_r[key]
        excess = (rejections - expected_rejections) / n_trials
        z_value = (rejections - expected_rejections) / sqrt(null_variance) if null_variance else 0.0
        p_value = normal_survival(z_value) if z_value > 0 else 1.0
        choice_rate = chosen_sessions[key] / seen_sessions[key] if seen_sessions[key] else 0.0
        scored.append({
            "key": key,
            "name": most_common_label(display_names[key], "Unknown"),
            "artist": most_common_label(display_artists[key], "Unknown"),
            "trials": n_trials,
            "rejections": rejections,
            "observed_rate": rejections / n_trials,
            "baseline_rate": expected_rejections / n_trials,
            "excess_rate": excess,
            "z_value": z_value,
            "p_value": p_value,
            "choice_rate": choice_rate,
            "score": max(excess, 0.0) * (1.0 - choice_rate),
        })

    benjamini_hochberg(scored)
    return scored, global_rejection_rate


def rank_candidates(rows, *, entity):
    if entity == "track":
        min_trials = CONFIG["track_min_exposures"]
        min_rejections = CONFIG["track_min_rejections"]
        min_excess = CONFIG["track_min_excess_rate"]
    else:
        min_trials = CONFIG["artist_min_exposures"]
        min_rejections = CONFIG["artist_min_rejections"]
        min_excess = CONFIG["artist_min_excess_rate"]

    candidates = []
    for row in rows:
        row["qualifies"] = (
            row["trials"] >= min_trials
            and row["rejections"] >= min_rejections
            and row["excess_rate"] >= min_excess
            and row["q_value"] <= CONFIG["max_false_discovery_rate"]
        )
        candidates.append(row)

    return sorted(
        (row for row in candidates if row["qualifies"]),
        key=lambda row: (-row["score"], -row["z_value"], -row["trials"], row["name"]),
    )


def print_candidates(title, rows, include_breadth=False, limit=10):
    print(f"\n{title}")
    if not rows:
        print("  No entity meets the predeclared support, excess, and FDR thresholds.")
        return
    header = "  #  Name / artist                              score   rejected / trials  excess  choice  q"
    print(header)
    print("  " + "-" * (len(header) - 2))
    for rank, row in enumerate(rows[:limit], start=1):
        name = row["name"] if not row["artist"] or row["name"] == row["artist"] else f"{row['name']} - {row['artist']}"
        suffix = f"  tracks={row['rejected_tracks']}" if include_breadth else ""
        print(
            f"  {rank:>2}  {name[:42]:<42} "
            f"{row['score']:>6.1%}  {row['rejections']:>3}/{row['trials']:<3} "
            f"{row['excess_rate']:>7.1%} {row['choice_rate']:>6.1%} {row['q_value']:.3g}{suffix}"
        )


track_scores, global_track_rejection_rate = score_contextual_rejection(
    track_trials, track_seen_sessions, track_chosen_sessions
)
artist_scores, global_artist_rejection_rate = score_contextual_rejection(
    artist_trials, artist_seen_sessions, artist_chosen_sessions
)

for row in artist_scores:
    row["rejected_tracks"] = len(rejected_track_keys_by_artist.get(row["key"], set()))
artist_candidates_before_breadth = rank_candidates(artist_scores, entity="artist")
artist_candidates = [
    row for row in artist_candidates_before_breadth
    if row["rejected_tracks"] >= CONFIG["artist_min_tracks"]
]
track_candidates = rank_candidates(track_scores, entity="track")

print(f"Global track-session rejection rate: {global_track_rejection_rate:.1%}")
print(f"Global artist-session rejection rate: {global_artist_rejection_rate:.1%}")
print(f"Track hypotheses screened: {len(track_scores):,}")
print(f"Artist hypotheses screened: {len(artist_scores):,}")
print_candidates("Qualified track candidates", track_candidates)
print_candidates(
    "Qualified multi-track artist candidates",
    artist_candidates,
    include_breadth=True,
)

if artist_candidates_before_breadth and not artist_candidates:
    strongest = artist_candidates_before_breadth[0]
    print(
        "\nArtist verdict withheld: the strongest artist-associated signal is "
        f"{strongest['name']}, but it only spans {strongest['rejected_tracks']} rejected track(s)."
    )

Global track-session rejection rate: 20.7%
Global artist-session rejection rate: 20.4%
Track hypotheses screened: 1,832
Artist hypotheses screened: 585

Qualified track candidates
  #  Name / artist                              score   rejected / trials  excess  choice  q
  ------------------------------------------------------------------------------------------
   1  If You Want Love - NF                       34.8%    7/11    47.4%  26.7% 0.000227
   2  Laugh Now Cry Later (feat. Lil Durk) - Dra  31.1%   13/22    34.5%  10.0% 0.00038
   3  Bloom - Aqyila                              25.6%   11/18    35.8%  28.6% 0.00207
   4  p r i d e . i s . t h e . d e v i l (with   22.7%   14/29    24.8%   8.3% 0.0065
   5  The Night We Met - Lord Huron               22.7%   13/24    28.4%  20.0% 0.00727
   6  Session 32 - Summer Walker                  14.2%   19/59    19.0%  25.0% 0.000227

Qualified multi-track artist candidates
  No entity meets the predeclared support, excess, and FDR thres

## 6. Artist Model: Pool Track-Level Evidence, Then Require Breadth

The artist-session version above is intentionally strict, but it can be too blunt: completing one song by an artist in a session should not erase a rapid rejection of a different song by that artist.

The artist model therefore reuses the independent **track-session** trials, pools their contextual evidence by artist, and excludes the whole artist from each matched baseline. It still uses direct artist-session starts as counterevidence and adds a breadth requirement: at least two distinct tracks must produce a rejection. The earlier artist-session result is retained as a conservative cross-check.

In [11]:
artist_track_trials = []
for trial in track_trials:
    pooled_trial = dict(trial)
    pooled_trial["entity_key"] = trial["artist_key"]
    artist_track_trials.append(pooled_trial)

artist_pooled_scores, global_artist_pooled_rejection_rate = score_contextual_rejection(
    artist_track_trials,
    artist_seen_sessions,
    artist_chosen_sessions,
)
for row in artist_pooled_scores:
    row["rejected_tracks"] = len(rejected_track_keys_by_artist.get(row["key"], set()))

artist_pooled_candidates_before_breadth = rank_candidates(artist_pooled_scores, entity="artist")
artist_pooled_candidates = [
    row for row in artist_pooled_candidates_before_breadth
    if row["rejected_tracks"] >= CONFIG["artist_min_tracks"]
]

print(f"Pooled track-session artist rejection rate: {global_artist_pooled_rejection_rate:.1%}")
print_candidates(
    "Qualified multi-track artist candidates (pooled track-session evidence)",
    artist_pooled_candidates,
    include_breadth=True,
)

if artist_pooled_candidates_before_breadth and not artist_pooled_candidates:
    strongest = artist_pooled_candidates_before_breadth[0]
    print(
        "\nArtist verdict withheld: the strongest pooled signal is "
        f"{strongest['name']}, but it only spans {strongest['rejected_tracks']} rejected track(s)."
    )

Pooled track-session artist rejection rate: 20.7%

Qualified multi-track artist candidates (pooled track-session evidence)
  #  Name / artist                              score   rejected / trials  excess  choice  q
  ------------------------------------------------------------------------------------------
   1  21 Savage                                   13.2%   15/39    16.9%  21.8% 0.0226  tracks=5


## 7. Robustness Checks

The 30% threshold is a transparent modeling choice, not a fact of nature. The notebook checks whether the leading candidates remain leading when “rapid” is defined as a forward skip before 20%, 25%, 30%, 35%, or 40% completion.

It then bootstraps whole listening sessions. Resampling by session preserves within-sitting dependence, including skip-through behavior and direct choices. A result is only called stable when it wins a substantial share of bootstrap reruns.

In [12]:
def track_trials_at_threshold(rejection_completion):
    """Rebuild labels at a new duration-normalized rejection cutoff."""
    original_labels = [play["rapid_rejection"] for play in metadata_plays]
    try:
        for play in metadata_plays:
            play["rapid_rejection"] = (
                play["passively_served"]
                and play["reason_end"] == "fwdbtn"
                and play["completion"] <= rejection_completion
            )
        return make_session_trials(track_session_events)
    finally:
        for play, label in zip(metadata_plays, original_labels):
            play["rapid_rejection"] = label


def pooled_artist_candidates(track_trial_set, artist_seen, artist_chosen):
    pooled_trials = []
    rejected_tracks = defaultdict(set)
    for trial in track_trial_set:
        pooled = dict(trial)
        pooled["entity_key"] = trial["artist_key"]
        pooled_trials.append(pooled)
        if trial["label"]:
            rejected_tracks[trial["artist_key"]].add(trial["track_key"])

    scores, _ = score_contextual_rejection(pooled_trials, artist_seen, artist_chosen)
    for row in scores:
        row["rejected_tracks"] = len(rejected_tracks[row["key"]])
    qualified = rank_candidates(scores, entity="artist")
    return [row for row in qualified if row["rejected_tracks"] >= CONFIG["artist_min_tracks"]]


threshold_results = []
for threshold in (0.20, 0.25, 0.30, 0.35, 0.40):
    threshold_track_trials, threshold_seen, threshold_chosen = track_trials_at_threshold(threshold)
    threshold_track_scores, _ = score_contextual_rejection(
        threshold_track_trials, threshold_seen, threshold_chosen
    )
    threshold_track_candidates = rank_candidates(threshold_track_scores, entity="track")
    threshold_artist_candidates = pooled_artist_candidates(
        threshold_track_trials, artist_seen_sessions, artist_chosen_sessions
    )
    threshold_results.append({
        "threshold": threshold,
        "track": threshold_track_candidates[0] if threshold_track_candidates else None,
        "artist": threshold_artist_candidates[0] if threshold_artist_candidates else None,
    })

print("Sensitivity to the definition of an early rejection:")
print("  cutoff   top qualified track                         top qualified multi-track artist")
print("  " + "-" * 86)
for result in threshold_results:
    track = result["track"]
    artist = result["artist"]
    track_label = f"{track['name']} - {track['artist']}" if track else "no qualified result"
    artist_label = artist["name"] if artist else "no qualified result"
    print(f"  {result['threshold']:>5.0%}   {track_label[:43]:<43} {artist_label[:35]}")

stable_track_thresholds = Counter(
    result["track"]["key"] for result in threshold_results if result["track"]
)
stable_artist_thresholds = Counter(
    result["artist"]["key"] for result in threshold_results if result["artist"]
)
print("\nThreshold wins:")
print("  Tracks: " + ", ".join(f"{key}: {count}/5" for key, count in stable_track_thresholds.most_common(3)))
print("  Artists: " + ", ".join(f"{key}: {count}/5" for key, count in stable_artist_thresholds.most_common(3)))

Sensitivity to the definition of an early rejection:
  cutoff   top qualified track                         top qualified multi-track artist
  --------------------------------------------------------------------------------------
    20%   If You Want Love - NF                       no qualified result
    25%   If You Want Love - NF                       21 Savage
    30%   If You Want Love - NF                       21 Savage
    35%   If You Want Love - NF                       21 Savage
    40%   LET GO - Central Cee                        21 Savage

Threshold wins:
  Tracks: 3i5qVV8azKqGFK4Gzdt5YS: 4/5, 3zkyus0njMCL6phZmNNEeN: 1/5
  Artists: id:1URnnhqYAYcrqrcwql10ft: 4/5


### Session Bootstrap

A bootstrap resamples the 1,095 complete listening sessions with replacement 250 times. Each resample rebuilds the trials and reruns the same score, support, false-discovery, direct-choice, and multi-track rules.

The bootstrap reports the share of reruns won by each candidate. This is not a literal probability that somebody hates a song; it is a stability check for the **ranking procedure** under plausible reweightings of the listener’s sessions.

In [13]:
session_events = defaultdict(list)
for play in metadata_plays:
    session_events[play["session"]].append(play)
original_session_ids = sorted(session_events)


def bootstrap_winners(random_generator):
    """Resample complete listening sessions and rerun the full primary model."""
    boot_track_groups = defaultdict(list)
    boot_artist_groups = defaultdict(list)

    for boot_session, source_session in enumerate(
        random_generator.choices(original_session_ids, k=len(original_session_ids))
    ):
        for event in session_events[source_session]:
            copied = dict(event)
            copied["session"] = boot_session
            boot_track_groups[(boot_session, copied["track_key"])].append(copied)
            boot_artist_groups[(boot_session, copied["artist_key"])].append(copied)

    boot_track_trials, boot_track_seen, boot_track_chosen = make_session_trials(boot_track_groups)
    _, boot_artist_seen, boot_artist_chosen = make_session_trials(boot_artist_groups)

    boot_track_scores, _ = score_contextual_rejection(
        boot_track_trials, boot_track_seen, boot_track_chosen
    )
    boot_tracks = rank_candidates(boot_track_scores, entity="track")
    boot_artists = pooled_artist_candidates(
        boot_track_trials, boot_artist_seen, boot_artist_chosen
    )
    return (
        boot_tracks[0] if boot_tracks else None,
        boot_artists[0] if boot_artists else None,
    )


random_generator = random.Random(CONFIG["random_seed"])
bootstrap_track_wins = Counter()
bootstrap_artist_wins = Counter()
bootstrap_track_labels = {}
bootstrap_artist_labels = {}
no_track_winner = 0
no_artist_winner = 0

for _ in range(CONFIG["bootstrap_samples"]):
    track_winner, artist_winner = bootstrap_winners(random_generator)
    if track_winner:
        bootstrap_track_wins[track_winner["key"]] += 1
        bootstrap_track_labels[track_winner["key"]] = (
            f"{track_winner['name']} - {track_winner['artist']}"
        )
    else:
        no_track_winner += 1
    if artist_winner:
        bootstrap_artist_wins[artist_winner["key"]] += 1
        bootstrap_artist_labels[artist_winner["key"]] = artist_winner["name"]
    else:
        no_artist_winner += 1


def print_bootstrap(title, wins, labels, no_winner):
    total = CONFIG["bootstrap_samples"]
    print(f"\n{title}")
    if not wins:
        print(f"  No qualified winner in any of {total} resamples.")
        return
    for rank, (key, count) in enumerate(wins.most_common(5), start=1):
        print(f"  {rank}. {labels[key]}: {count}/{total} ({count / total:.1%})")
    if no_winner:
        print(f"  No qualified winner: {no_winner}/{total} ({no_winner / total:.1%})")


print(f"Resampling {len(original_session_ids):,} complete listening sessions {CONFIG['bootstrap_samples']:,} times...")
print_bootstrap("Bootstrap track-winner stability", bootstrap_track_wins, bootstrap_track_labels, no_track_winner)
print_bootstrap("Bootstrap multi-track artist-winner stability", bootstrap_artist_wins, bootstrap_artist_labels, no_artist_winner)

Resampling 1,081 complete listening sessions 250 times...

Bootstrap track-winner stability
  1. You Are The Reason - Calum Scott: 49/250 (19.6%)
  2. If You Want Love - NF: 43/250 (17.2%)
  3. Laugh Now Cry Later (feat. Lil Durk) - Drake: 32/250 (12.8%)
  4. Yosemite (Song For The Ahwahnechee) - Iniko: 13/250 (5.2%)
  5. Birr Indaygezash - Sami Dan: 11/250 (4.4%)

Bootstrap multi-track artist-winner stability
  1. 21 Savage: 64/250 (25.6%)
  2. NF: 64/250 (25.6%)
  3. Billie Eilish: 37/250 (14.8%)
  4. Eyob Mekonen: 23/250 (9.2%)
  5. Roddy Ricch: 14/250 (5.6%)
  No qualified winner: 9/250 (3.6%)


## 8. Alternative Model: Long-Term Recurring Rejection

The bootstrap says that several tracks are close under the session model. This alternative makes a stricter long-term claim:

- one track gets at most one outcome per calendar day;
- a direct selection anywhere that day excludes that day from passive-exposure evidence;
- a durable listen anywhere that day overrides a rapid skip;
- the same contextual excess, false-discovery, direct-choice, and artist-breadth rules are reused.

This method intentionally discounts a track that was rejected many times in one day. It answers a slightly different question: *which track or artist is repeatedly rejected across separate days?*

In [14]:
def make_calendar_day_trials(grouped_events):
    """Collapse one entity's events within a calendar day into one conservative label."""
    trials = []
    seen_days = Counter()
    chosen_days = Counter()

    for (day, entity_key), events in grouped_events.items():
        seen_days[entity_key] += 1
        if any(event["chosen"] for event in events):
            chosen_days[entity_key] += 1
            continue

        passive_events = [event for event in events if event["passively_served"]]
        durable_events = [event for event in passive_events if event["durable_acceptance"]]
        rejection_events = [event for event in passive_events if event["rapid_rejection"]]
        if durable_events:
            label = 0
            anchor = max(durable_events, key=lambda event: (event["completion"], event["timestamp"]))
        elif rejection_events:
            label = 1
            anchor = min(rejection_events, key=lambda event: event["timestamp"])
        else:
            continue

        trial = dict(anchor)
        trial["day"] = day
        trial["entity_key"] = entity_key
        trial["label"] = label
        trial["event_count"] = len(events)
        trials.append(trial)

    return trials, seen_days, chosen_days


track_day_events = defaultdict(list)
artist_day_events = defaultdict(list)
for play in metadata_plays:
    day = play["timestamp"].date().isoformat()
    track_day_events[(day, play["track_key"])].append(play)
    artist_day_events[(day, play["artist_key"])].append(play)

track_day_trials, track_seen_days, track_chosen_days = make_calendar_day_trials(track_day_events)
_, artist_seen_days, artist_chosen_days = make_calendar_day_trials(artist_day_events)

recurring_track_scores, recurring_track_rate = score_contextual_rejection(
    track_day_trials, track_seen_days, track_chosen_days
)
recurring_track_candidates = rank_candidates(recurring_track_scores, entity="track")

recurring_artist_trials = []
recurring_rejected_track_keys_by_artist = defaultdict(set)
for trial in track_day_trials:
    pooled_trial = dict(trial)
    pooled_trial["entity_key"] = trial["artist_key"]
    recurring_artist_trials.append(pooled_trial)
    if trial["label"]:
        recurring_rejected_track_keys_by_artist[trial["artist_key"]].add(trial["track_key"])

recurring_artist_scores, recurring_artist_rate = score_contextual_rejection(
    recurring_artist_trials, artist_seen_days, artist_chosen_days
)
for row in recurring_artist_scores:
    row["rejected_tracks"] = len(recurring_rejected_track_keys_by_artist[row["key"]])
recurring_artist_candidates = [
    row for row in rank_candidates(recurring_artist_scores, entity="artist")
    if row["rejected_tracks"] >= CONFIG["artist_min_tracks"]
]

assert len({(trial["day"], trial["entity_key"]) for trial in track_day_trials}) == len(track_day_trials)
print(f"Track-day trials: {len(track_day_trials):,} ({sum(trial['label'] for trial in track_day_trials):,} rejections)")
print(f"Global track-day rejection rate: {recurring_track_rate:.1%}")
print_candidates("Qualified long-term track candidates", recurring_track_candidates)
print_candidates(
    "Qualified long-term multi-track artist candidates",
    recurring_artist_candidates,
    include_breadth=True,
)

Track-day trials: 7,365 (1,520 rejections)
Global track-day rejection rate: 20.6%

Qualified long-term track candidates
  #  Name / artist                              score   rejected / trials  excess  choice  q
  ------------------------------------------------------------------------------------------
   1  Laugh Now Cry Later (feat. Lil Durk) - Dra  31.3%   13/22    34.9%  10.3% 0.000405
   2  If You Want Love - NF                       30.9%    6/10    43.2%  28.6% 0.00127
   3  The Night We Met - Lord Huron               23.3%   13/24    28.5%  18.2% 0.00704
   4  p r i d e . i s . t h e . d e v i l (with   23.2%   13/27    25.5%   8.8% 0.00675

Qualified long-term multi-track artist candidates
  #  Name / artist                              score   rejected / trials  excess  choice  q
  ------------------------------------------------------------------------------------------
   1  21 Savage                                   13.8%   13/35    18.0%  23.4% 0.0145  tracks=5


### Calendar-Day Bootstrap

The recurring-rejection model gets its own matching bootstrap: it resamples complete calendar days with replacement, preserving all same-day events and direct selections. Its reported winner share is therefore a stability check for the day-level model, while the earlier session bootstrap remains a check for the session-context alternative.

In [16]:
calendar_day_events = defaultdict(list)
for play in metadata_plays:
    calendar_day_events[play["timestamp"].date().isoformat()].append(play)
original_calendar_days = sorted(calendar_day_events)


def bootstrap_recurring_winners(random_generator):
    """Resample complete calendar days and rerun the recurring-rejection model."""
    boot_track_groups = defaultdict(list)
    boot_artist_groups = defaultdict(list)

    for boot_day, source_day in enumerate(
        random_generator.choices(original_calendar_days, k=len(original_calendar_days))
    ):
        for event in calendar_day_events[source_day]:
            copied = dict(event)
            boot_track_groups[(boot_day, copied["track_key"])].append(copied)
            boot_artist_groups[(boot_day, copied["artist_key"])].append(copied)

    boot_track_trials, boot_track_seen, boot_track_chosen = make_calendar_day_trials(boot_track_groups)
    _, boot_artist_seen, boot_artist_chosen = make_calendar_day_trials(boot_artist_groups)
    boot_track_scores, _ = score_contextual_rejection(
        boot_track_trials, boot_track_seen, boot_track_chosen
    )
    boot_tracks = rank_candidates(boot_track_scores, entity="track")
    boot_artists = pooled_artist_candidates(
        boot_track_trials, boot_artist_seen, boot_artist_chosen
    )
    return (
        boot_tracks[0] if boot_tracks else None,
        boot_artists[0] if boot_artists else None,
    )


recurring_random_generator = random.Random(CONFIG["random_seed"] + 1)
recurring_bootstrap_track_wins = Counter()
recurring_bootstrap_artist_wins = Counter()
recurring_bootstrap_track_labels = {}
recurring_bootstrap_artist_labels = {}
no_recurring_track_winner = 0
no_recurring_artist_winner = 0

for _ in range(CONFIG["bootstrap_samples"]):
    track_winner, artist_winner = bootstrap_recurring_winners(recurring_random_generator)
    if track_winner:
        recurring_bootstrap_track_wins[track_winner["key"]] += 1
        recurring_bootstrap_track_labels[track_winner["key"]] = (
            f"{track_winner['name']} - {track_winner['artist']}"
        )
    else:
        no_recurring_track_winner += 1
    if artist_winner:
        recurring_bootstrap_artist_wins[artist_winner["key"]] += 1
        recurring_bootstrap_artist_labels[artist_winner["key"]] = artist_winner["name"]
    else:
        no_recurring_artist_winner += 1

print(
    f"Resampling {len(original_calendar_days):,} complete calendar days "
    f"{CONFIG['bootstrap_samples']:,} times..."
)
print_bootstrap(
    "Bootstrap recurring-track winner stability",
    recurring_bootstrap_track_wins,
    recurring_bootstrap_track_labels,
    no_recurring_track_winner,
)
print_bootstrap(
    "Bootstrap recurring multi-track artist-winner stability",
    recurring_bootstrap_artist_wins,
    recurring_bootstrap_artist_labels,
    no_recurring_artist_winner,
)

Resampling 432 complete calendar days 250 times...

Bootstrap recurring-track winner stability
  1. You Are The Reason - Calum Scott: 45/250 (18.0%)
  2. Laugh Now Cry Later (feat. Lil Durk) - Drake: 32/250 (12.8%)
  3. If You Want Love - NF: 19/250 (7.6%)
  4. Don't Let Me Down - Conor Maynard: 12/250 (4.8%)
  5. Arcade - Duncan Laurence: 10/250 (4.0%)

Bootstrap recurring multi-track artist-winner stability
  1. 21 Savage: 71/250 (28.4%)
  2. NF: 47/250 (18.8%)
  3. Billie Eilish: 19/250 (7.6%)
  4. Eyob Mekonen: 18/250 (7.2%)
  5. Ed Sheeran: 15/250 (6.0%)
  No qualified winner: 21/250 (8.4%)


## 9. Decision, Metadata Dossiers, and Export

The models answer related but not identical questions:

1. **Session-context model:** Which track has the strongest excess of rapid passive skips once same-session repetition and direct choice are controlled?  
2. **Recurring-rejection model:** Which track is rejected most strongly across separate calendar days?

The recurring-rejection model is the recommended product definition because it discounts a one-day or one-session skip spiral. The session-context result remains a useful alternative. The artist verdict uses the recurring model plus the two-track breadth rule.

A bootstrap found that the exact top *track* changes across resampled sessions, so the report deliberately calls this a supported recommendation and named shortlist, not a psychological fact. Metadata describes the finalists; it never supplies the dislike label.

In [17]:
import csv


def bootstrap_share(row, win_counts):
    if not row:
        return 0.0
    return win_counts[row["key"]] / CONFIG["bootstrap_samples"]


def track_metadata_dossier(row):
    metadata = metadata_by_id.get(row["key"], {})
    return {
        "track_id": row["key"],
        "track": row["name"],
        "artist": row["artist"],
        "duration_seconds": metadata.get("duration_s"),
        "release_year": metadata.get("release_year"),
        "genres": metadata.get("genres"),
        "popularity": metadata.get("popularity"),
        "energy": metadata.get("energy"),
        "valence": metadata.get("valence"),
    }


def artist_metadata_dossier(row):
    artist_events = [play for play in metadata_plays if play["artist_key"] == row["key"]]
    genre_counts = Counter(play["genres"] for play in artist_events if play["genres"])
    years = [play["release_year"] for play in artist_events if play["release_year"]]
    popularity = [play["popularity"] for play in artist_events if play["popularity"] is not None]
    return {
        "artist": row["name"],
        "catalog_genres": most_common_label(genre_counts, None),
        "catalog_release_year_range": [min(years), max(years)] if years else None,
        "catalog_mean_popularity": round(statistics.mean(popularity), 3) if popularity else None,
        "catalog_matched_events": len(artist_events),
    }


def concise_dossier(dossier):
    duration = dossier.get("duration_seconds")
    duration_label = f"{duration // 60}:{duration % 60:02d}" if duration else "unknown"
    popularity = dossier.get("popularity")
    popularity_label = f"{popularity:.2f}" if popularity is not None else "unknown"
    return (
        f"duration={duration_label}; release={dossier.get('release_year') or 'unknown'}; "
        f"genres={dossier.get('genres') or 'unknown'}; popularity={popularity_label}"
    )


session_track_winner = track_candidates[0] if track_candidates else None
recurring_track_winner = recurring_track_candidates[0] if recurring_track_candidates else None
recurring_artist_winner = recurring_artist_candidates[0] if recurring_artist_candidates else None

session_track_stability = bootstrap_share(session_track_winner, bootstrap_track_wins)
recurring_track_stability = bootstrap_share(
    recurring_track_winner, recurring_bootstrap_track_wins
)
recurring_artist_stability = bootstrap_share(
    recurring_artist_winner, recurring_bootstrap_artist_wins
)

print("FINAL DECISION")
print("=" * 70)
if recurring_track_winner:
    print("Recommended long-term recurring-rejection result:")
    print(
        f"  {recurring_track_winner['name']} - {recurring_track_winner['artist']} "
        f"({recurring_track_winner['rejections']}/{recurring_track_winner['trials']} independent days rejected; "
        f"{recurring_track_winner['excess_rate']:.1%} contextual excess; "
        f"{recurring_track_winner['choice_rate']:.1%} direct-choice rate)"
    )
    print(f"  Catalog context: {concise_dossier(track_metadata_dossier(recurring_track_winner))}")
    print(f"  Calendar-day bootstrap winner share: {recurring_track_stability:.1%}")

if session_track_winner:
    print("\nAlternative session-context result:")
    print(
        f"  {session_track_winner['name']} - {session_track_winner['artist']} "
        f"({session_track_winner['rejections']}/{session_track_winner['trials']} independent sessions rejected; "
        f"{session_track_winner['excess_rate']:.1%} contextual excess; "
        f"{session_track_winner['choice_rate']:.1%} direct-choice rate)"
    )
    print(f"  Catalog context: {concise_dossier(track_metadata_dossier(session_track_winner))}")
    print(f"  Session-bootstrap winner share: {session_track_stability:.1%}")

if recurring_artist_winner:
    artist_catalog = artist_metadata_dossier(recurring_artist_winner)
    print("\nRecommended multi-track artist result:")
    print(
        f"  {recurring_artist_winner['name']} "
        f"({recurring_artist_winner['rejections']}/{recurring_artist_winner['trials']} independent track-days rejected; "
        f"{recurring_artist_winner['rejected_tracks']} distinct rejected tracks; "
        f"{recurring_artist_winner['excess_rate']:.1%} contextual excess; "
        f"{recurring_artist_winner['choice_rate']:.1%} direct-choice rate)"
    )
    print(
        "  Catalog context: "
        f"genres={artist_catalog['catalog_genres'] or 'unknown'}; "
        f"releases={artist_catalog['catalog_release_year_range'] or 'unknown'}; "
        f"mean popularity={artist_catalog['catalog_mean_popularity'] or 'unknown'}"
    )
    print(f"  Calendar-day bootstrap winner share: {recurring_artist_stability:.1%}")

print("\nInterpretation:")
print(
    "  The evidence supports a ranked rejection shortlist, not a literal proof of emotion. "
    "No point winner has majority stability in the matching bootstrap, so keep both track definitions visible "
    "instead of claiming a uniquely certain winner."
)

report = {
    "model": "contextual_rejection_v1",
    "generated_from": [path.name for path in source_files],
    "events": {
        "raw_music_events": len(music_rows),
        "deduplicated_music_events": len(plays),
        "metadata_duration_events": len(metadata_plays),
        "metadata_duration_coverage": len(metadata_plays) / len(plays),
        "metadata_covered_listening_sessions": len(original_session_ids),
        "metadata_covered_calendar_days": len(original_calendar_days),
    },
    "configuration": {
        "rapid_completion": CONFIG["rapid_completion"],
        "session_gap_minutes": CONFIG["session_gap_minutes"],
        "prior_strength": CONFIG["prior_strength"],
        "bootstrap_samples": CONFIG["bootstrap_samples"],
        "direct_choice_starts": sorted(CONFIG["choice_starts"]),
    },
    "verdict": {
        "recommended_definition": "recurring_rejection_across_calendar_days",
        "recommended_track": recurring_track_winner,
        "session_context_track_alternative": session_track_winner,
        "recommended_multitrack_artist": recurring_artist_winner,
        "unique_track_winner_supported": bool(recurring_track_winner and recurring_track_stability >= 0.50),
        "unique_artist_winner_supported": bool(recurring_artist_winner and recurring_artist_stability >= 0.50),
        "note": "Point rankings are statistically qualified, but no winner has majority stability under its matching bootstrap.",
    },
    "metadata_dossiers": {
        "recurring_track": track_metadata_dossier(recurring_track_winner) if recurring_track_winner else None,
        "session_track": track_metadata_dossier(session_track_winner) if session_track_winner else None,
        "artist": artist_metadata_dossier(recurring_artist_winner) if recurring_artist_winner else None,
    },
    "robustness": {
        "threshold_wins": {
            "tracks": dict(stable_track_thresholds),
            "artists": dict(stable_artist_thresholds),
        },
        "session_bootstrap_wins": {
            "tracks": {
                bootstrap_track_labels[key]: count
                for key, count in bootstrap_track_wins.most_common()
            },
            "artists": {
                bootstrap_artist_labels[key]: count
                for key, count in bootstrap_artist_wins.most_common()
            },
        },
        "calendar_day_bootstrap_wins": {
            "tracks": {
                recurring_bootstrap_track_labels[key]: count
                for key, count in recurring_bootstrap_track_wins.most_common()
            },
            "artists": {
                recurring_bootstrap_artist_labels[key]: count
                for key, count in recurring_bootstrap_artist_wins.most_common()
            },
        },
        "winner_stability": {
            "session_context_track": session_track_stability,
            "recurring_track": recurring_track_stability,
            "recurring_multitrack_artist": recurring_artist_stability,
        },
    },
}

report_path = EXPORT_DIR / "contextual_rejection_report.json"
with report_path.open("w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2, ensure_ascii=True)

ranking_path = EXPORT_DIR / "contextual_rejection_rankings.csv"
ranking_fields = [
    "model", "rank", "name", "artist", "trials", "rejections", "observed_rate",
    "baseline_rate", "excess_rate", "choice_rate", "score", "z_value", "p_value",
    "q_value", "rejected_tracks", "qualifies",
]
ranking_sets = [
    ("session_track", track_candidates),
    ("recurring_track", recurring_track_candidates),
    ("recurring_artist", recurring_artist_candidates),
]
with ranking_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=ranking_fields)
    writer.writeheader()
    for model_name, rows in ranking_sets:
        for rank, row in enumerate(rows, start=1):
            writer.writerow({
                field: row.get(field, "") for field in ranking_fields
            } | {"model": model_name, "rank": rank})

print(f"\nWrote audit report: {report_path}")
print(f"Wrote ranked candidates: {ranking_path}")

FINAL DECISION
Recommended long-term recurring-rejection result:
  Laugh Now Cry Later (feat. Lil Durk) - Drake (13/22 independent days rejected; 34.9% contextual excess; 10.3% direct-choice rate)
  Catalog context: duration=4:21; release=2020; genres=hip hop, canadian hip hop, rap, pop rap, canadian pop; popularity=0.76
  Calendar-day bootstrap winner share: 12.8%

Alternative session-context result:
  If You Want Love - NF (7/11 independent sessions rejected; 47.4% contextual excess; 26.7% direct-choice rate)
  Catalog context: duration=3:19; release=2017; genres=hip hop, pop rap; popularity=0.72
  Session-bootstrap winner share: 17.2%

Recommended multi-track artist result:
  21 Savage (13/35 independent track-days rejected; 5 distinct rejected tracks; 18.0% contextual excess; 23.4% direct-choice rate)
  Catalog context: genres=rap, atl hip hop, hip hop; releases=[2018, 2024]; mean popularity=0.781
  Calendar-day bootstrap winner share: 28.4%

Interpretation:
  The evidence supports